# Wave propagation: Fresnel and Fraunhofer diffraction

![Fig. 8](../.figure/fig8.jpg)

## 1. Coordinate system and basic concepts

Consider the free-space propagation of a wave. Let the aperture plane be the $z=0$ plane, and denote the coordinates on it by

$$
(\xi,\eta)
$$

The observation plane is a plane located a distance $z$ from the aperture plane, and its coordinates are denoted by

$$
(x,y)
$$

Let the complex amplitude immediately after the aperture plane be

$$
u_0(\xi,\eta)
$$

and write the complex amplitude at point $P$ after propagation by a distance $z$ as

$$
u_P(x,y;z)
$$

The scalar wavenumber is $k=\frac{2\pi}{\lambda}$. Here, $\lambda$ is the wavelength.

The distance from a point $(\xi,\eta,0)$ on the aperture plane to the observation point $P(x,y,z)$ is

$$
r=\sqrt{z^2+(x-\xi)^2+(y-\eta)^2}
$$

According to Huygens' principle, each point on the aperture plane acts like a point source that emits a secondary spherical wave. Therefore, the field on the observation plane is given by the superposition of the spherical waves emitted from all points on the aperture plane.

However, it is more accurate to distinguish the following cases.

- At short distances, the phase differences among the spherical waves emitted from each point source must be considered directly. This region is the Fresnel diffraction region.
- At sufficiently long distances, the wavefront formed by the point sources can be regarded as a plane wave, and the phase variations of the plane waves formed by the point sources become almost linear for each observation direction. This region is the Fraunhofer diffraction region.

---


In [ ]:
# Section 1: numerical grid, aperture field, and physical constants
import numpy as np
import matplotlib.pyplot as plt

wavelength = 633e-9          # wavelength [m]
k = 2 * np.pi / wavelength   # wavenumber [rad/m]
N = 512                      # grid samples per dimension
L = 5.0e-3                   # side length of aperture window [m]
dx = L / N
x = (np.arange(N) - N // 2) * dx
X, Y = np.meshgrid(x, x, indexing="xy")

# Example aperture: a square opening with a small circular phase object.
aperture_width = 0.7e-3
u0 = ((np.abs(X) <= aperture_width / 2) & (np.abs(Y) <= aperture_width / 2)).astype(complex)
phase_object = np.exp(1j * 0.8 * np.exp(-(X**2 + Y**2) / (0.18e-3)**2))
u0 *= phase_object

plt.figure(figsize=(4, 3.5))
plt.imshow(np.abs(u0), extent=[x[0]*1e3, x[-1]*1e3, x[0]*1e3, x[-1]*1e3], cmap="gray")
plt.xlabel("x [mm]")
plt.ylabel("y [mm]")
plt.title("Aperture amplitude |u0|")
plt.colorbar(label="amplitude")
plt.tight_layout()


## 2. Huygens-Kirchhoff integral

Let us ignore the process by which the wave was formed before the aperture plane, and assume that the complex amplitude $u_0(\xi,\eta)$ immediately after the aperture plane is given.

One form of the Huygens-Kirchhoff diffraction integral can be written as

$$
u(x,y;z)
=
-\frac{ik}{4\pi}
\iint
u_0(\xi,\eta)
\frac{e^{ikr}}{r}
\left(1+\cos\theta\right)
\,d\xi d\eta .
$$

Here, $\theta$ is the angle between the normal direction of the aperture plane and the observation direction. Also, since

$$
k=\frac{2\pi}{\lambda}
$$

the expression above can be written as

$$
u(x,y;z)
=
-\frac{i}{2\lambda}
\iint
u_0(\xi,\eta)
\frac{e^{ikr}}{r}
\left(1+\cos\theta\right)
\,d\xi d\eta
$$

If the obliquity factor is defined as

$$
K(\theta)=\frac{1+\cos\theta}{2}
$$

then

$$
u(x,y;z)
=
-\frac{i}{\lambda}
\iint
u_0(\xi,\eta)
\frac{e^{ikr}}{r}
K(\theta)
\,d\xi d\eta .
$$

Under the paraxial approximation, that is, when the observation angle $\theta$ is sufficiently small,

$$
\cos\theta \approx 1
$$

so

$$
K(\theta)\approx 1
$$

Therefore,

$$
u(x,y;z)
\approx
-\frac{i}{\lambda}
\iint
u_0(\xi,\eta)
\frac{e^{ikr}}{r}
\,d\xi d\eta .
$$

Also, the $r$ in the amplitude term varies slowly compared with the phase term $e^{ikr}$, so we can set

$$
r \approx z
$$

Then

$$
u(x,y;z)
\approx
-\frac{i}{\lambda z}
\iint
u_0(\xi,\eta)
e^{ikr}
\,d\xi d\eta .
$$

The important point here is as follows.

The $r$ in the denominator, which corresponds to amplitude, can be approximated by $z$, but the $r$ inside the exponential must not be replaced by $z$ carelessly. This is because in optics $k=2\pi/\lambda$ is very large, so even a very small path-length difference can produce a large phase difference.

---


In [ ]:
# Section 2: direct Huygens-Kirchhoff / Rayleigh-Sommerfeld-style reference integral
# This direct integral is O(N^4), so it is evaluated on a small grid for demonstration.
def huygens_kirchhoff_direct(u0, aperture_axis, obs_axis, z, wavelength):
    """Direct paraxial scalar diffraction integral with r kept in the phase."""
    k = 2 * np.pi / wavelength
    dxi = aperture_axis[1] - aperture_axis[0]
    XI, ETA = np.meshgrid(aperture_axis, aperture_axis, indexing="xy")
    out = np.empty((len(obs_axis), len(obs_axis)), dtype=complex)
    for iy, yy in enumerate(obs_axis):
        for ix, xx in enumerate(obs_axis):
            r = np.sqrt(z**2 + (xx - XI)**2 + (yy - ETA)**2)
            out[iy, ix] = (-1j / wavelength) * np.sum(u0 * np.exp(1j * k * r) / r) * dxi**2
    return out

# Down-sample the aperture to keep the direct reference inexpensive.
small_step = 16
u0_small = u0[::small_step, ::small_step]
x_small = x[::small_step]
obs_axis_small = np.linspace(-1.0e-3, 1.0e-3, 33)
z_reference = 0.08
u_hk_small = huygens_kirchhoff_direct(u0_small, x_small, obs_axis_small, z_reference, wavelength)
I_hk_small = np.abs(u_hk_small)**2
I_hk_small /= I_hk_small.max()

plt.figure(figsize=(4, 3.5))
plt.imshow(I_hk_small, extent=[obs_axis_small[0]*1e3, obs_axis_small[-1]*1e3, obs_axis_small[0]*1e3, obs_axis_small[-1]*1e3], cmap="magma")
plt.xlabel("x [mm]")
plt.ylabel("y [mm]")
plt.title("Direct integral reference intensity")
plt.colorbar(label="normalized intensity")
plt.tight_layout()


## 3. Taylor expansion of the distance $r$

Now approximate the quantity inside the exponential,

$$
r=\sqrt{z^2+(x-\xi)^2+(y-\eta)^2}
$$

First, it can be written as

$$
r
=
z
\sqrt{
1+
\frac{(x-\xi)^2+(y-\eta)^2}{z^2}
}
$$

Using

$$
\sqrt{1+s}
=
1+\frac{s}{2}-\frac{s^2}{8}+\cdots
$$

gives

$$
r
=
z
\left[
1
+
\frac{1}{2}
\frac{(x-\xi)^2+(y-\eta)^2}{z^2}
-
\frac{1}{8}
\left(
\frac{(x-\xi)^2+(y-\eta)^2}{z^2}
\right)^2
+
\cdots
\right].
$$

Therefore,

$$
r
\approx
z
+
\frac{(x-\xi)^2+(y-\eta)^2}{2z}
$$

One should note that terms cannot be discarded merely because the later terms in the Taylor expansion appear small. For the Fresnel approximation to be valid, the phase error caused by the neglected terms,

$$
k\Delta r
$$

must be sufficiently small.

Substituting this approximation into the Huygens-Kirchhoff integral gives

$$
e^{ikr}
\approx
e^{ikz}
\exp
\left[
i\frac{k}{2z}
\left\{
(x-\xi)^2+(y-\eta)^2
\right\}
\right].
$$

Therefore,

$$
u(x,y;z)
\approx
-\frac{i e^{ikz}}{\lambda z}
\iint
u_0(\xi,\eta)
\exp
\left[
i\frac{k}{2z}
\left\{
(x-\xi)^2+(y-\eta)^2
\right\}
\right]
\,d\xi d\eta .
$$

Since $k=2\pi/\lambda$,

$$
\frac{k}{2z}
=
\frac{\pi}{\lambda z}
$$

Thus the Fresnel diffraction integral is

$$
u(x,y;z)
\approx
-\frac{i e^{ikz}}{\lambda z}
\iint
u_0(\xi,\eta)
\exp
\left[
i\frac{\pi}{\lambda z}
\left\{
(x-\xi)^2+(y-\eta)^2
\right\}
\right]
\,d\xi d\eta .
$$

Equivalently, using

$$
\frac{1}{i}=-i
$$

we can write the same expression as

$$
u(x,y;z)
\approx
\frac{e^{ikz}}{i\lambda z}
\iint
u_0(\xi,\eta)
\exp
\left[
i\frac{\pi}{\lambda z}
\left\{
(x-\xi)^2+(y-\eta)^2
\right\}
\right]
\,d\xi d\eta
$$

---


In [ ]:
# Section 3: Fresnel propagation as convolution with the quadratic impulse response
# This implements the approximation derived from the Taylor expansion of r.
def fft2c(a):
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(a)))

def ifft2c(a):
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(a)))

def fresnel_transfer_propagate(u0, dx, z, wavelength):
    """Fresnel propagation using the transfer-function form on the same output grid."""
    n = u0.shape[0]
    fx = np.fft.fftshift(np.fft.fftfreq(n, d=dx))
    FX, FY = np.meshgrid(fx, fx, indexing="xy")
    k = 2 * np.pi / wavelength
    H = np.exp(1j * k * z) * np.exp(-1j * np.pi * wavelength * z * (FX**2 + FY**2))
    return ifft2c(fft2c(u0) * H)

z_fresnel_demo = 0.08
u_fresnel_demo = fresnel_transfer_propagate(u0, dx, z_fresnel_demo, wavelength)
I_fresnel_demo = np.abs(u_fresnel_demo)**2
I_fresnel_demo /= I_fresnel_demo.max()

plt.figure(figsize=(4, 3.5))
plt.imshow(I_fresnel_demo, extent=[x[0]*1e3, x[-1]*1e3, x[0]*1e3, x[-1]*1e3], cmap="magma")
plt.xlabel("x [mm]")
plt.ylabel("y [mm]")
plt.title(f"Fresnel intensity, z={z_fresnel_demo:.2f} m")
plt.colorbar(label="normalized intensity")
plt.tight_layout()


## 4. Fourier-transform form of the Fresnel diffraction integral

Now expand the square inside the exponential.

$$
(x-\xi)^2+(y-\eta)^2
=
x^2+y^2+\xi^2+\eta^2-2x\xi-2y\eta .
$$

Therefore,

$$
u(x,y;z)
\approx
\frac{e^{ikz}}{i\lambda z}
\iint
u_0(\xi,\eta)
\exp
\left[
i\frac{\pi}{\lambda z}
\left(
x^2+y^2+\xi^2+\eta^2-2x\xi-2y\eta
\right)
\right]
\,d\xi d\eta .
$$

The terms containing only $x$ and $y$ are independent of the integration variables $\xi,\eta$, so they can be taken outside the integral.

$$
u(x,y;z)
\approx
\frac{e^{ikz}}{i\lambda z}
\exp
\left[
i\frac{\pi}{\lambda z}
(x^2+y^2)
\right]
\iint
u_0(\xi,\eta)
\exp
\left[
i\frac{\pi}{\lambda z}
(\xi^2+\eta^2)
\right]
\exp
\left[
-i\frac{2\pi}{\lambda z}
(x\xi+y\eta)
\right]
\,d\xi d\eta .
$$

This is an important form of the Fresnel diffraction integral.

Define the Fourier transform of an arbitrary function $g(\xi,\eta)$ as

$$
\mathcal{F}\{g(\xi,\eta)\}(f_x,f_y)
=
\iint
g(\xi,\eta)
e^{-i2\pi(f_x\xi+f_y\eta)}
\,d\xi d\eta .
$$

Then Fresnel diffraction can be regarded as the Fourier transform of

$$
g(\xi,\eta)
=
u_0(\xi,\eta)
\exp
\left[
i\frac{\pi}{\lambda z}
(\xi^2+\eta^2)
\right]
$$

In this case, the spatial frequencies of the Fourier transform are

$$
f_x=\frac{x}{\lambda z},
\qquad
f_y=\frac{y}{\lambda z}
$$

Therefore, Fresnel diffraction is not simply a Fourier transform. Rather, it is the Fourier transform obtained after multiplying

$$
u_0(\xi,\eta)
$$

by the quadratic phase term (chirp phase)

$$
\exp
\left[
i\frac{\pi}{\lambda z}
(\xi^2+\eta^2)
\right]
$$

---


In [ ]:
# Section 4: Fourier-transform form of Fresnel diffraction
# The output coordinates are determined by fx=x/(lambda*z), so x_out=lambda*z*fx.
def fresnel_fourier_form(u0, dx, z, wavelength):
    n = u0.shape[0]
    aperture_axis = (np.arange(n) - n // 2) * dx
    XI, ETA = np.meshgrid(aperture_axis, aperture_axis, indexing="xy")
    fx = np.fft.fftshift(np.fft.fftfreq(n, d=dx))
    x_out = wavelength * z * fx
    XOUT, YOUT = np.meshgrid(x_out, x_out, indexing="xy")
    k = 2 * np.pi / wavelength
    chirped_input = u0 * np.exp(1j * np.pi / (wavelength * z) * (XI**2 + ETA**2))
    G = fft2c(chirped_input) * dx**2
    prefactor = np.exp(1j * k * z) / (1j * wavelength * z)
    output_chirp = np.exp(1j * np.pi / (wavelength * z) * (XOUT**2 + YOUT**2))
    return prefactor * output_chirp * G, x_out

z_fresnel = 0.25
u_fresnel, x_fresnel = fresnel_fourier_form(u0, dx, z_fresnel, wavelength)
I_fresnel = np.abs(u_fresnel)**2
I_fresnel /= I_fresnel.max()

plt.figure(figsize=(4, 3.5))
plt.imshow(I_fresnel, extent=[x_fresnel[0]*1e3, x_fresnel[-1]*1e3, x_fresnel[0]*1e3, x_fresnel[-1]*1e3], cmap="magma")
plt.xlabel("x [mm]")
plt.ylabel("y [mm]")
plt.title(f"Fresnel propagation (Fourier form), z={z_fresnel:.2f} m")
plt.colorbar(label="normalized intensity")
plt.tight_layout()


## 5. Fraunhofer diffraction

Fraunhofer diffraction is a further approximation of Fresnel diffraction.

Inside the Fresnel integral, there is the following quadratic phase term on the aperture-plane side:

$$
\exp
\left[
i\frac{\pi}{\lambda z}
(\xi^2+\eta^2)
\right].
$$

In the Fraunhofer approximation, the phase variation of this term is assumed to be sufficiently small over the entire aperture plane.

That is, if

$$
\frac{\pi}{\lambda z}(\xi^2+\eta^2) \ll 1
$$

then

$$
\exp
\left[
i\frac{\pi}{\lambda z}
(\xi^2+\eta^2)
\right]
\approx 1
$$

If the representative aperture size is $D$, the Fraunhofer approximation is roughly valid when

$$
z \gg \frac{D^2}{\lambda}
$$

Applying this approximation gives

$$
u_F(x,y;z)
\approx
\frac{e^{ikz}}{i\lambda z}
\exp
\left[
i\frac{\pi}{\lambda z}
(x^2+y^2)
\right]
\iint
u_0(\xi,\eta)
\exp
\left[
-i\frac{2\pi}{\lambda z}
(x\xi+y\eta)
\right]
\,d\xi d\eta .
$$

Comparing this with the definition of the Fourier transform,

$$
u_F(x,y;z)
\approx
\frac{e^{ikz}}{i\lambda z}
\exp
\left[
i\frac{\pi}{\lambda z}
(x^2+y^2)
\right]
\mathcal{F}\{u_0(\xi,\eta)\}
\left(
\frac{x}{\lambda z},
\frac{y}{\lambda z}
\right).
$$

Thus, in Fraunhofer diffraction, the complex amplitude on the observation plane is proportional to the Fourier transform of the complex amplitude on the aperture plane.

The observed intensity is

$$
I_F(x,y;z)
=
|u_F(x,y;z)|^2
$$

so the pure phase factor multiplied in front does not affect the intensity. Therefore, the intensity pattern in the Fraunhofer region is given by

$$
I_F(x,y;z)
\propto
\left|
\mathcal{F}\{u_0(\xi,\eta)\}
\left(
\frac{x}{\lambda z},
\frac{y}{\lambda z}
\right)
\right|^2
$$

However, in this relation the coordinates of the Fourier transform are not the physical observation coordinates $(x,y)$, but the spatial-frequency coordinates scaled as

$$
\left(
\frac{x}{\lambda z},
\frac{y}{\lambda z}
\right)
$$


In [ ]:
# Section 5: Fraunhofer propagation as the Fourier transform of the aperture field
def fraunhofer_propagate(u0, dx, z, wavelength):
    n = u0.shape[0]
    fx = np.fft.fftshift(np.fft.fftfreq(n, d=dx))
    x_out = wavelength * z * fx
    XOUT, YOUT = np.meshgrid(x_out, x_out, indexing="xy")
    k = 2 * np.pi / wavelength
    U0 = fft2c(u0) * dx**2
    prefactor = np.exp(1j * k * z) / (1j * wavelength * z)
    output_chirp = np.exp(1j * np.pi / (wavelength * z) * (XOUT**2 + YOUT**2))
    return prefactor * output_chirp * U0, x_out

z_far = 5.0
u_fraunhofer, x_far = fraunhofer_propagate(u0, dx, z_far, wavelength)
I_fraunhofer = np.abs(u_fraunhofer)**2
I_fraunhofer /= I_fraunhofer.max()

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
axes[0].imshow(I_fraunhofer, extent=[x_far[0]*1e3, x_far[-1]*1e3, x_far[0]*1e3, x_far[-1]*1e3], cmap="magma")
axes[0].set_title(f"Fraunhofer intensity, z={z_far:.1f} m")
axes[0].set_xlabel("x [mm]")
axes[0].set_ylabel("y [mm]")

center = I_fraunhofer.shape[0] // 2
axes[1].plot(x_far * 1e3, I_fraunhofer[center])
axes[1].set_xlim(-20, 20)
axes[1].set_xlabel("x [mm]")
axes[1].set_ylabel("normalized intensity")
axes[1].set_title("Central cross-section")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
